In [24]:
from extract import extract_graph , extract_die_area , load_file_content

# Define your file paths
file_name = "aes_run_20260305_181833"
def_path = f"./CTS-Bench/runs/{file_name}/11-openroad-detailedplacement/aes.def"
saif_path = f"./CTS-Bench/runs/{file_name}/aes.saif"
timing_path = f"./CTS-Bench/runs/{file_name}/timing_paths.csv"

# One function call to get everything!
graph = extract_graph(def_path, saif_path, timing_path, clock_port="clk")
def_text = load_file_content(def_path)
die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)



print("\nExtraction Complete!")
print(f"Nodes: {len(graph['nodes'])}")
print(f"Skip edges: {graph['skip_edges'].shape}")
print(f"Directed edges: {graph['directed_edges'].shape}")
print(f"Undirected edges: {graph['undirected_edges'].shape}")
print(f"Die area: x[{die_x_min}, {die_x_max}], y[{die_y_min}, {die_y_max}]")    

for a, b in graph['skip_edges'][:5]:  # Print first 5 skip edges
    print(f"Skip edge: {a, b}")

Flip-flops: 2994
Unique pairs: 5596, dropped: 0
Skip edges shape: (5596, 2)
Nodes: 24364
Directed edges: 77018
Undirected edges: 154036
Flip-flops: 2994
Fan-in  — max: 6, avg: 3.2
Fan-out — max: 153, avg: 3.2

Extraction Complete!
Nodes: 24364
Skip edges: (5596, 2)
Directed edges: (77018, 2)
Undirected edges: (154036, 2)
Die area: x[0, 540775], y[0, 1081235]
Skip edge: (np.int64(20751), np.int64(23414))
Skip edge: (np.int64(20751), np.int64(23415))
Skip edge: (np.int64(20751), np.int64(23412))
Skip edge: (np.int64(20751), np.int64(23452))
Skip edge: (np.int64(20751), np.int64(23605))


In [25]:
import numpy as np 
import torch

def normalize_features(nodes, die_x_min, die_y_min, die_x_max, die_y_max):
    die_w = die_x_max - die_x_min
    die_h = die_y_max - die_y_min
    
    cell_ids = []
    numeric = []
    
    for n in nodes:
        row = [
            (n['x'] - die_x_min) / die_w,
            (n['y'] - die_y_min) / die_h,
            n['dist_to_boundaries'][0] / die_w,
            n['dist_to_boundaries'][1] / die_w,
            n['dist_to_boundaries'][2] / die_h,
            n['dist_to_boundaries'][3] / die_h,
            np.log1p(n['cell_area']),
            n['avg_pin_cap'] * 1000,
            n['total_pin_cap'] * 1000,
            np.log2(max(n['drive_strength'], 1)),
            float(n['is_sequential']),
            float(n['is_buffer']),
            n['toggle_count'],
            n['sum_toggle_count'],
            n['signal_prob'],
            float(n['non_zero_count']),
            np.log1p(n['fan_in']),
            np.log1p(n['fan_out']),
        ]
        numeric.append(row)
        cell_ids.append(n['cell_type_id'])
    
    features = torch.tensor(numeric, dtype=torch.float32)
    cell_type_ids = torch.tensor(cell_ids, dtype=torch.long)
    
    # Standardize all columns except binary flags (indices 10, 11)
    binary_cols = {10, 11}
    norm_stats = {}
    
    for col in range(features.shape[1]):
        if col in binary_cols:
            continue
        mean = features[:, col].mean()
        std = features[:, col].std()
        if std > 1e-8:
            features[:, col] = (features[:, col] - mean) / std
            norm_stats[col] = (mean.item(), std.item())
    
    return features, cell_type_ids, norm_stats

# Assuming 'graph' and 'die_x_min' etc. are already defined from previous cells
X_float, X_cell_ids, norm_stats = normalize_features(
    graph['nodes'], 
    die_x_min, die_y_min, die_x_max, die_y_max
)

print("-" * 30)
print("NORMALIZATION RESULTS")
print("-" * 30)
print(f"X_float shape:    {X_float.shape} (Standardized continuous features)")
print(f"X_cell_ids shape: {X_cell_ids.shape} (Categorical IDs for embeddings)")
print(f"Number of normalized columns: {len(norm_stats)}")

# Sample check on first node
print("\nFirst Node Normalized Features:")
print(X_float[17])
print(X_cell_ids[17])


# Verify the categorical ID
print(f"\nFirst Node Cell Type ID: {X_cell_ids[0].item()}")
print(f"\nSecond Node Cell Type ID: {X_cell_ids[1045].item()}")

------------------------------
NORMALIZATION RESULTS
------------------------------
X_float shape:    torch.Size([24364, 18]) (Standardized continuous features)
X_cell_ids shape: torch.Size([24364]) (Categorical IDs for embeddings)
Number of normalized columns: 16

First Node Normalized Features:
tensor([-0.5958,  1.0806, -0.5958,  0.5958, -1.0806,  1.0806,  0.7999,  8.2356,
         1.6574,  3.3326,  0.0000,  1.0000,  0.9242,  0.8937,  0.4033, -1.5944,
        -2.3439,  3.2434])
tensor(199)

First Node Cell Type ID: 201

Second Node Cell Type ID: 20


In [ ]:
import scipy.sparse as sp
import numpy as np

def build_X_hop_mask(n_nodes, undirected_edges, hop_mask_len=3):
    """
    Creates the Omega mask. 
    If Omega[i, j] = 1, Node i is allowed to 'use' Node j for reconstruction.
    """
    # 1. Slice out the Source (rows) and Destination (cols) nodes
    u_rows = undirected_edges[:, 0]
    u_cols = undirected_edges[:, 1]
    
    # 2. Build the initial 1-hop Sparse Adjacency Matrix (A)
    # Using bool to save memory (we only care IF a wire exists)
    A = sp.csr_matrix((np.ones(len(u_rows), dtype=bool), (u_rows, u_cols)), 
                      shape=(n_nodes, n_nodes))
    
    # 3. Matrix Power Expansion (Find 2-hop and 3-hop neighbors)
    omega = A.copy()
    temp = A.copy()
    
    for i in range(2, hop_mask_len + 1):
        temp = temp.dot(A)
        omega = omega + temp
        print(f"  -> Hop {i} expansion complete.")

    # 4) diag(P) = 0
    # A node cannot reconstruct itself
    omega.setdiag(0)
    omega.eliminate_zeros()
    
    # 5. Convert back to coordinates for PyTorch
    omega_coo = omega.tocoo()
    
    print(f"Mask created! {omega_coo.nnz:,} total connections allowed.")
    return omega_coo.row, omega_coo.col


# Pass the length and the edges explicitly 
p_rows, p_cols = build_X_hop_mask(
    n_nodes=len(graph['nodes']), 
    undirected_edges=graph['undirected_edges'], 
    hop_mask_len=3
)


# # 1. Create a reverse dictionary to turn indices back into DEF instance names
# idx_to_node = {idx: name for name, idx in graph['node_to_idx'].items()}

# # 2. Check the overall density (How heavy is this for the GPU?)
# total_edges = len(p_rows)
# total_nodes = len(graph['nodes'])
# avg_neighbors = total_edges / total_nodes

# print("--- MASK STATISTICS ---")
# print(f"Total Nodes (Gates): {total_nodes:,}")
# print(f"Total Allowed Connections in Mask: {total_edges:,}")
# print(f"Average 3-Hop Neighbors per Gate: {avg_neighbors:.1f}")

# # 3. Peek at the actual relationships (The first 10 connections)
# print("\n--- SAMPLE CONNECTIONS (Who can see who?) ---")
# for i in range(10):
#     target_idx = p_rows[i]
#     neighbor_idx = p_cols[i]
    
#     target_name = idx_to_node[target_idx]
#     neighbor_name = idx_to_node[neighbor_idx]
    
#     print(f"Target Gate [{target_idx:5}]: {target_name:<20} <-- looks at --> Neighbor [{neighbor_idx:5}]: {neighbor_name}")

# # 4. Look at a specific gate's neighborhood
# sample_gate_idx = p_rows[0] # Let's pick whatever gate happens to be first
# sample_gate_name = idx_to_node[sample_gate_idx]

# # Find all neighbors for this specific gate
# neighbors_of_sample = p_cols[p_rows == sample_gate_idx]

# print(f"\n--- SPOTLIGHT: Gate '{sample_gate_name}' ---")
# print(f"This gate has {len(neighbors_of_sample)} neighbors within 3 hops.")
# print(f"First 5 neighbors it will use for reconstruction:")
# for n_idx in neighbors_of_sample[:5]:
#     print(f"  - {idx_to_node[n_idx]}")



  -> Hop 2 expansion complete.
  -> Hop 3 expansion complete.
Mask created! 10,922,930 total connections allowed.
--- MASK STATISTICS ---
Total Nodes (Gates): 24,364
Total Allowed Connections in Mask: 10,922,930
Average 3-Hop Neighbors per Gate: 448.3

--- SAMPLE CONNECTIONS (Who can see who?) ---
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20652]: _41111_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20663]: _41122_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20669]: _41128_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20680]: _41139_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20684]: _41143_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20685]: _41144_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20735]: _41194_
Target Gate [    0]: _20459_              <-- looks at --> Neighbor [20751]: _41210_
Target Gate [    0]: 

In [54]:
import torch.optim as optim
class FirstTerm(nn.Module):
    def __init__(self, indices, num_nodes , num_cell_types, embedding_dim=8):
        super().__init__()
        self.register_buffer('indices', indices)
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        self.p_weights = nn.Parameter(torch.randn(indices.shape[1]) * 0.01)
        self.num_nodes = num_nodes
        self.proj = nn.Linear(18+ embedding_dim, 18)  

    def forward(self, X, X_cell_ids):
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]
        X_proj = self.proj(X_combined)  # Shape: [num_nodes, 18]


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(self.indices, torch.relu(self.p_weights), 
                                    (self.num_nodes, self.num_nodes))
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X_proj)
        
        # Loss: ||X - XP||
        error = X_proj - X_hat
        loss = torch.norm(error, p='fro')
        return loss

# --- 3. THE INTEGRATED EXECUTION (The Run) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_nodes = len(graph['nodes'])

# Step A: Build the mask
# 1. Get the two lists (rows and columns) from your function
rows, cols = build_X_hop_mask(num_nodes, graph['undirected_edges'], hop_mask_len=3)

# 2. Stack them into one Tensor and then move to device
p_indices = torch.tensor(np.array([rows, cols]), dtype=torch.long).to(device)

num_cell_types = X_cell_ids.max().item() + 1 

# Step B: Initialize Model & Data
model = FirstTerm(p_indices, num_nodes, num_cell_types).to(device)
X_tensor = X_float.to(device) # Your normalized features
cell_ids_tensor = X_cell_ids.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Step C: The Optimization Loop
print(f"\nTraining on {device}...")
for epoch in range(51):
    optimizer.zero_grad()
    loss = model(X_tensor, cell_ids_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2} | Reconstruction Loss: {loss.item():.4f}")

print("\nSuccess! Your manifold is trained. Your chip data is now organized.")

  -> Hop 2 expansion complete.
  -> Hop 3 expansion complete.
Mask created! 10,922,930 total connections allowed.

Training on cpu...
Epoch  0 | Reconstruction Loss: 408.5621
Epoch 10 | Reconstruction Loss: 51.1967
Epoch 20 | Reconstruction Loss: 20.1778
Epoch 30 | Reconstruction Loss: 9.4996
Epoch 40 | Reconstruction Loss: 4.6943
Epoch 50 | Reconstruction Loss: 3.0577

Success! Your manifold is trained. Your chip data is now organized.


In [55]:
# After training, check if the projection is collapsing
X_enriched = model.proj(torch.cat([X_tensor, model.cell_embedding(cell_ids_tensor)], dim=1))
print(X_enriched.std(dim=0))  # if many dims are near zero, it's collapsing

tensor([0.0547, 0.0603, 0.0235, 0.0304, 0.0276, 0.0388, 0.0526, 0.0238, 0.0860,
        0.0310, 0.0157, 0.0228, 0.0147, 0.0462, 0.0369, 0.0505, 0.0352, 0.0342],
       grad_fn=<StdBackward0>)
